# Attribution Demo

In this demo, you'll learn how to load models and perform attribution on them using [circuit-tracer](https://github.com/decoderesearch/circuit-tracer).

In [3]:
# Install circuit-tracer from GitHub (not on PyPI)
!pip install git+https://github.com/decoderesearch/circuit-tracer.git



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: Could not find a version that satisfies the requirement circuit-tracer (from versions: none)

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for circuit-tracer


In [1]:
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files

ModuleNotFoundError: No module named 'torch'

## Load Model

Load your model and transcoders by name. `model_name` is a HuggingFace / TransformerLens model name.

Supported models:
- `google/gemma-2-2b` with `"gemma"` transcoders (Gemma Scope, lowest L0)
- `meta-llama/Llama-3.2-1B` with `"llama"` transcoders (ReLU skip-transcoders)

For other models, provide your own transcoder config file.

In [ ]:
model_name = "google/gemma-2-2b"
transcoder_name = "gemma"
backend = "transformerlens"  # change to 'nnsight' for the nnsight backend
model = ReplacementModel.from_pretrained(
    model_name, transcoder_name, dtype=torch.bfloat16, backend=backend
)

## Set Attribution Arguments

In [ ]:
prompt = "The capital of state containing Dallas is"
max_n_logits = 10
desired_logit_prob = 0.95
max_feature_nodes = 8192
batch_size = 256
offload = "cpu"  # 'disk', 'cpu', or None (keep on GPU)
verbose = True

## Run Attribution

In [ ]:
graph = attribute(
    prompt=prompt,
    model=model,
    max_n_logits=max_n_logits,
    desired_logit_prob=desired_logit_prob,
    batch_size=batch_size,
    max_feature_nodes=max_feature_nodes,
    offload=offload,
    verbose=verbose,
)

## Save Graph

Save the graph as a `.pt` file (can be large, ~167MB).

In [ ]:
graph_dir = Path("graphs")
graph_dir.mkdir(exist_ok=True)
graph_path = graph_dir / "example_graph.pt"

graph.to_pt(graph_path)

## Create Visualization Files

Create the graph files needed for the interactive visualization. Pruning thresholds control how aggressively low-impact nodes/edges are removed:
- `node_threshold=0.8` — keep minimum nodes to capture 80% of logit effects
- `edge_threshold=0.98` — keep minimum edges to capture 98% of logit effects

In [ ]:
slug = "dallas-austin"
graph_file_dir = "./graph_files"
node_threshold = 0.8
edge_threshold = 0.98

create_graph_files(
    graph_or_path=graph_path,
    slug=slug,
    output_path=graph_file_dir,
    node_threshold=node_threshold,
    edge_threshold=edge_threshold,
)

## Visualize the Graph

Spin up a local server for the interactive frontend.

**If running on a remote server, set up port forwarding so the chosen port is accessible locally.**

Interactions:
- Click nodes to select them
- Ctrl/Cmd+Click to pin/unpin nodes to subgraph
- G+Click to group nodes into a supernode
- G+Click on X to dissolve a supernode

In [ ]:
from circuit_tracer.frontend.local_server import serve
from IPython.display import IFrame

port = 8046
server = serve(data_dir="./graph_files/", port=port)

print(f"Open your graph here: http://localhost:{port}/index.html")
display(IFrame(src=f"http://localhost:{port}/index.html", width="100%", height="800px"))

In [ ]:
# Stop the server when done
# server.stop()

## Optional: Prune Graph Further

You can re-prune with different thresholds to get smaller/larger circuits.

In [ ]:
from circuit_tracer.graph import prune_graph

prune_graph(graph, node_threshold=0.7, edge_threshold=0.95)